# Pawn counting

Project to explore/visualise chess pawn structures

In [165]:
from pathlib import Path
import polars as pl
import numpy as np
import altair as alt

## basic functions

In [166]:
type Position = tuple[np.uint64, np.uint64]
"""
Pawn structure position --- a configuration of White and Black pawns on a board of size
BOARD_WIDTH * BOARD_DEPTH

Consists of two u64 bitmasks, the first for White pawns and the second for Black.

The least bit of each bitmask is the bottom-left square for White (a1), counting up in
file-major order (a1, a2, ..., a8, b1, ...) to the top-right square for White (h8).

If the board is smaller than 8x8, the board crops around bottom-left square for White
(so the bottom-left square remains a1), and the unused squares are empty.
"""

BOARD_WIDTH = 4
BOARD_DEPTH = 4

assert 0 <= BOARD_WIDTH <= 8
assert 0 <= BOARD_DEPTH <= 8

In [167]:
def position_to_int(pos: Position) -> int:
    """Pack a Position into a single UInt128: low 64 bits White, high 64 bits Black."""
    white, black = pos
    return int(white) | (int(black) << 64)

def position_from_int(code: int) -> Position:
    """Unpack a UInt128 (low bits White, high bits Black) back into a Position."""
    mask = (1 << 64) - 1
    return np.uint64(code & mask), np.uint64((code >> 64) & mask)

In [168]:
def position_ndarray(pos: Position) -> tuple[np.ndarray, np.ndarray]:
    def to_array(mask):
        arr = np.zeros((BOARD_WIDTH, BOARD_DEPTH), dtype=bool)
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                arr[f, r] = (mask >> (f * 8 + r)) & 1
        return arr

    white, black = pos
    return to_array(white), to_array(black)

def init_position() -> Position:
    white = 0
    black = 0
    for f in range(BOARD_WIDTH):
        white |= 1 << (f * 8)
        black |= 1 << (f * 8 + BOARD_DEPTH - 1)
    return np.uint64(white), np.uint64(black)

def rand_position(max_pawns_per_side: int | None = None) -> Position:
    squares = [f * 8 + r for f in range(BOARD_WIDTH) for r in range(BOARD_DEPTH)]
    n = len(squares)
    cap = n if max_pawns_per_side is None else min(max_pawns_per_side, n)

    n_white = np.random.randint(cap + 1)
    n_black = np.random.randint(min(cap, n - n_white) + 1)

    chosen = np.random.choice(n, size=n_white + n_black, replace=False)
    white = 0
    black = 0
    for i in chosen[:n_white]:
        white |= 1 << squares[i]
    for i in chosen[n_white:]:
        black |= 1 << squares[i]
    return np.uint64(white), np.uint64(black)

def pawns_as_frame(pos: Position) -> pl.DataFrame:
    colour = pl.Enum(["White", "Black"])
    rows = []
    for name, mask in zip(["White", "Black"], pos):
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                if (mask >> (f * 8 + r)) & 1:
                    rows.append({"rank": r + 1, "file": f + 1, "colour": name})
    return pl.DataFrame(rows, schema={"rank": pl.Int64, "file": pl.Int64, "colour": colour})

In [169]:
def position_chart(pos: Position, *, axis=False, legend=False) -> alt.LayerChart:
    _PAWN_COLOURS = {
        "domain": ["White", "Black"],
        "range": ["white", "black"],
    }
    _SQUARE_COLOURS = {
        "domain": ["light", "dark"],
        "range": ["#f0d9b5", "#b58863"],
    }
    _FILE_DOMAIN = list(range(1, BOARD_WIDTH + 1))
    _RANK_DOMAIN = list(range(1, BOARD_DEPTH + 1))

    def _board_as_frame() -> pl.DataFrame:
        rows = []
        for f in range(1, BOARD_WIDTH + 1):
            for r in range(1, BOARD_DEPTH + 1):
                square = "dark" if (f + r) % 2 == 0 else "light"
                rows.append({"rank": r, "file": f, "square": square})
        return pl.DataFrame(rows)

    axis_params = {} if axis else {"axis": None}
    legend_params = {} if legend else {"legend": None}

    board = (
        alt.Chart(_board_as_frame())
        .mark_rect()
        .encode(
            alt.X("file:O", **axis_params).scale(domain=_FILE_DOMAIN),  # type: ignore
            alt.Y("rank:O", **axis_params).scale(domain=_RANK_DOMAIN),  # type: ignore
            alt.Color("square:N")  # type: ignore
            .scale(**_SQUARE_COLOURS)  # type: ignore
            .legend(None),
        )
    )

    pawns = (
        alt.Chart(pawns_as_frame(pos))
        .mark_circle(size=250, stroke="black", strokeWidth=0.5)
        .encode(
            alt.X("file:O").scale(domain=_FILE_DOMAIN),
            alt.Y("rank:O").scale(domain=_RANK_DOMAIN, reverse=True),
            alt.Color("colour:N", **legend_params)  # type: ignore
            #
            .scale(**_PAWN_COLOURS),  # type: ignore
        )
    )

    return (
        alt.layer(board, pawns)
        .resolve_scale(color="independent")
        .properties(width=150, height=150)
    )  # type: ignore


position_chart(init_position())

alt.LayerChart(...)

In [170]:
def _on_board(f: int, r: int) -> bool:
    return 0 <= f < BOARD_WIDTH and 0 <= r < BOARD_DEPTH

def accessible_positions(pos: Position) -> list[tuple[str, int, Position]]:
    white, black = int(pos[0]), int(pos[1])
    out: list[tuple[str, int, Position]] = []

    def emit(kind, src, new_own, new_enemy, white_to_move):
        w, b = (new_own, new_enemy) if white_to_move else (new_enemy, new_own)
        out.append((kind, src, (np.uint64(w), np.uint64(b))))

    for white_to_move in (True, False):
        own, enemy = (white, black) if white_to_move else (black, white)
        dr = 1 if white_to_move else -1
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                src = f * 8 + r
                if not (own >> src) & 1:
                    continue
                own_removed = own & ~(1 << src)

                af, ar = f, r + dr  # advance (straight)
                if _on_board(af, ar):
                    dst = af * 8 + ar
                    if not ((own | enemy) >> dst) & 1:
                        emit("A", src, own_removed | (1 << dst), enemy, white_to_move)
                else:
                    emit("A", src, own_removed, enemy, white_to_move)  # off end rank == R

                for kind, df in (("CQ", -1), ("CK", 1)):  # diagonal captures
                    cf, cr = f + df, r + dr
                    if not 0 <= cf < BOARD_WIDTH:
                        continue  # off the side: not a legal capture
                    if not 0 <= cr < BOARD_DEPTH:
                        emit(kind, src, own_removed, enemy, white_to_move)  # off end rank == R
                        continue
                    dst = cf * 8 + cr
                    if (own >> dst) & 1:
                        continue  # friendly block
                    emit(
                        kind,
                        src,
                        own_removed | (1 << dst),
                        enemy & ~(1 << dst),
                        white_to_move,
                    )

                emit("R", src, own_removed, enemy, white_to_move)  # always

    return out

In [171]:
def multiple_positions_chart(poses: list[Position], *, width: int | None = None) -> alt.VConcatChart:
    pos_charts = [position_chart(p) for p in poses]
    if width is None:
        width = int(np.sqrt(len(pos_charts)))
    return alt.vconcat(
        *(
            alt.hconcat(*pos_charts[n : (n + width)])
            for n in range(0, len(pos_charts), width)
        )
    )

multiple_positions_chart([p for _, _, p in accessible_positions(init_position())], width=7)

alt.VConcatChart(...)

## generation

In [172]:
def explore_transitions(start: Position) -> pl.DataFrame:
    """Every transition reachable from `start`, one row per transition.

    Positions are packed into a UInt128: low 64 bits White, high 64 bits Black.
    `transition_depth` is the minimum number of steps to reach the position the
    move is made from (the BFS level of `start_pos`).

    Edges are collected one BFS level at a time and concatenated, keeping peak
    memory bounded (~2.4 GB / ~2 min for the 4x4 start position: ~2.2M positions,
    ~46M transitions).
    """
    transition_type = pl.Enum(["A", "CQ", "CK", "R"])
    schema = {
        "start_pos": pl.UInt128,
        "end_pos": pl.UInt128,
        "moving_pawn": pl.UInt8,
        "transition_type": transition_type,
    }

    def encode(pos: Position) -> int:
        return int(pos[0]) | (int(pos[1]) << 64)

    seen = {encode(start)}
    frontier = [start]
    chunks: list[pl.DataFrame] = []

    depth = 0
    while frontier:
        starts, ends, pawns, kinds = [], [], [], []
        next_frontier = []
        for pos in frontier:
            start_code = encode(pos)
            for kind, src, end in accessible_positions(pos):
                end_code = encode(end)
                starts.append(start_code)
                ends.append(end_code)
                pawns.append(src)
                kinds.append(kind)
                if end_code not in seen:
                    seen.add(end_code)
                    next_frontier.append(end)
        if starts:
            chunks.append(
                pl.DataFrame(
                    {"start_pos": starts, "end_pos": ends,
                     "moving_pawn": pawns, "transition_type": kinds},
                    schema=schema,
                ).with_columns(transition_depth=pl.lit(depth, dtype=pl.UInt8))
            )
        frontier = next_frontier
        depth += 1

    return pl.concat(chunks, rechunk=False)


if not Path("./transitions.tmp.parquet").exists():
    _transitions = explore_transitions(init_position())
    _transitions.write_parquet("transitions.tmp.parquet")
    del _transitions
transitions = pl.read_parquet("transitions.tmp.parquet")
print(transitions.schema)
transitions.head(100)

Schema({'start_pos': UInt128, 'end_pos': UInt128, 'moving_pawn': UInt8, 'transition_type': Enum(categories=['A', 'CQ', 'CK', 'R']), 'transition_depth': UInt8})


start_pos,end_pos,moving_pawn,transition_type,transition_depth
u128,u128,u8,enum,u8
2485589411633493130050863361,2485589411633493130050863362,0,"""A""",0
2485589411633493130050863361,2485589411633493130050863872,0,"""CK""",0
2485589411633493130050863361,2485589411633493130050863360,0,"""R""",0
2485589411633493130050863361,2485589411633493130050863617,8,"""A""",0
2485589411633493130050863361,2485589411633493130050863107,8,"""CQ""",0
…,…,…,…,…
2485589411633493130050863360,2485589264059540540374450432,3,"""R""",1
2485589411633493130050863360,2485570522167561651470008576,11,"""A""",1
2485589411633493130050863360,2485551706488606467727360256,11,"""CQ""",1


## exploration

### transition depth

In [178]:
# branching factor plot

_plot_df = (
    transitions.group_by("transition_depth", "start_pos")
    .agg(transitions_count=pl.len())
    .group_by("transition_depth")
    .agg(
        transitions_count=pl.col("transitions_count").sum(),
        branching_factor=pl.col("transitions_count").mean(),
    )
    .sort("transition_depth")
)

alt.hconcat(
    alt.Chart(_plot_df).mark_line().encode(x="transition_depth", y="transitions_count"),
    alt.Chart(_plot_df).mark_line().encode(x="transition_depth", y="branching_factor"),
)

alt.HConcatChart(...)

In [ ]:
# positions leading to empty
multiple_positions_chart(
    [
        position_from_int(long_pos)
        for long_pos in transitions.filter(
            pl.col("end_pos") == 0, pl.col("transition_type") != "R"
        )["start_pos"]
    ],
    width=7,
)

alt.VConcatChart(...)